# Uncertainty FinBERT — GPU Training & Inference
Run this notebook on **Google Colab with A100 GPU + High-RAM**.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate datasets pyarrow pandas numpy tqdm scikit-learn

In [ ]:
# Cell 2: Mount Drive and copy files
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/data/processed'
MODEL_DIR = '/content/models/uncertainty_finbert'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Copy from your Google Drive
!cp "/content/drive/My Drive/Colab Notebooks/processed/"*.parquet /content/data/processed/
!ls -lh /content/data/processed/

# Verify files
for f in ['train.parquet', 'val.parquet', 'test.parquet', 'transcripts_parsed.parquet']:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✓ {f} ({size_mb:.1f} MB)')
    else:
        print(f'  ✗ {f} — MISSING!')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → A100 GPU')

In [ ]:
# Cell 4: Train FinBERT model (~10 min on A100)
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL_NAME = 'ProsusAI/finbert'

class SentenceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary'),
        'recall': recall_score(labels, preds, average='binary'),
    }

# Load data
train_df = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

train_dataset = SentenceDataset(train_df['sentence'].tolist(), train_df['binary_label'].tolist(), tokenizer)
val_dataset = SentenceDataset(val_df['sentence'].tolist(), val_df['binary_label'].tolist(), tokenizer)

n_train_steps = (len(train_dataset) // 32 + 1) * 4
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'), num_train_epochs=4,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01, warmup_steps=int(n_train_steps * 0.1),
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', greater_is_better=True, logging_steps=50,
    save_total_limit=2, fp16=True, report_to='none', seed=42,
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'Model saved to {MODEL_DIR}')
metrics = trainer.evaluate()
print(f'\nValidation Metrics: {metrics}')

In [ ]:
# Cell 5: Test evaluation
test_df = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))
test_dataset = SentenceDataset(test_df['sentence'].tolist(), test_df['binary_label'].tolist(), tokenizer)
test_metrics = trainer.evaluate(test_dataset)
print(f'\nTest Metrics: {test_metrics}')

In [ ]:
# Cell 6: Score ALL 140K transcripts (~15 min on A100)
import re, time
from tqdm import tqdm

SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
UNCERTAINTY_WORDS = {
    'approximate','approximately','assumption','assumptions','believe','believed','believes',
    'cautious','conceivable','conditional','contingency','contingent','could','depend',
    'dependent','depends','doubt','doubtful','estimate','estimated','estimates','estimating',
    'eventual','eventually','exposure','fluctuate','fluctuation','forecast','forecasts',
    'hypothetical','imprecise','indefinite','indefinitely','indeterminate','likelihood',
    'may','maybe','might','nearly','pending','perhaps','possibility','possible','possibly',
    'precaution','predict','predicted','prediction','preliminary','presumably','presumption',
    'probabilistic','probability','probable','probably','projected','projection','provisional',
    'random','reassess','revision','risky','rough','roughly','seems','sometime','sometimes',
    'somewhat','speculate','speculation','speculative','sudden','suddenly','suggest',
    'suggested','suggests','susceptible','tend','tendency','tends','tentative','uncertain',
    'uncertainties','uncertainty','unclear','undefined','unexpected','unexpectedly',
    'unforeseen','unknown','unplanned','unpredictable','unresolved','unsettled','unspecified',
    'unusual','vague','variability','variable','variance','variation','volatile','volatility',
}

def split_into_sentences(text):
    if not text or not text.strip(): return []
    return [s.strip() for s in SENTENCE_SPLIT_RE.split(text.strip()) if len(s.strip()) > 20]

def lexicon_score_text(text):
    if not text or not isinstance(text, str) or len(text.strip()) < 50: return np.nan
    sentences = split_into_sentences(text)
    if not sentences: return np.nan
    ratios = []
    for s in sentences:
        tokens = re.findall(r'\b[a-z]+\b', s.lower())
        ratios.append(sum(1 for t in tokens if t in UNCERTAINTY_WORDS) / max(len(tokens), 1) if len(tokens) >= 3 else 0.0)
    return float(np.mean(ratios))

# Load trained model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to('cuda')
model.eval()

# Load transcripts
df = pd.read_parquet(os.path.join(DATA_DIR, 'transcripts_parsed.parquet'))
print(f'Loaded {len(df)} transcripts')

# Phase 1: Split sentences
print('Phase 1: Splitting sentences...')
t0 = time.time()
rows = []
for i, row in tqdm(df.iterrows(), total=len(df), desc='Splitting'):
    tid = row['transcript_id']
    for section, col in [('full','management_text'),('prepared','prepared_remarks'),('qa','qa_text')]:
        text = row.get(col, '')
        if not text or not isinstance(text, str) or len(text.strip()) < 50: continue
        for sent in split_into_sentences(text):
            rows.append({'transcript_id': tid, 'section': section, 'sentence': sent})
sent_df = pd.DataFrame(rows)
del rows
print(f'Total sentences: {len(sent_df):,} ({time.time()-t0:.1f}s)')

# Phase 2: GPU inference
print('\nPhase 2: GPU inference...')
BATCH_SIZE_INF = 1024
all_sentences = sent_df['sentence'].tolist()
n = len(all_sentences)
probs = np.zeros(n, dtype=np.float32)
lengths = np.array([len(s) for s in all_sentences])
sorted_indices = np.argsort(lengths)
sorted_sentences = [all_sentences[i] for i in sorted_indices]

t1 = time.time()
with torch.inference_mode():
    for start in tqdm(range(0, n, BATCH_SIZE_INF), desc='Inference', unit='batch'):
        batch_texts = sorted_sentences[start:start+BATCH_SIZE_INF]
        encodings = tokenizer(batch_texts, truncation=True, padding='longest', max_length=128, return_tensors='pt')
        encodings = {k: v.to('cuda') for k, v in encodings.items()}
        with torch.autocast(device_type='cuda'):
            logits = model(**encodings).logits
        batch_probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        probs[sorted_indices[start:start+BATCH_SIZE_INF]] = batch_probs
print(f'\nInference done: {time.time()-t1:.0f}s ({n/(time.time()-t1):.0f} sent/sec)')

# Phase 3: Aggregate
print('\nPhase 3: Aggregating...')
sent_df['prob'] = probs
results = []
for tid, group in sent_df.groupby('transcript_id'):
    record = {'transcript_id': tid}
    for section, prefix in [('full',''),('prepared','prepared_'),('qa','qa_')]:
        sec = group[group['section']==section]
        if len(sec)==0:
            record[f'{prefix}uncertainty_score']=np.nan; record[f'{prefix}uncertainty_share']=np.nan
            if section=='full': record['n_sentences']=0
        else:
            p=sec['prob'].values; record[f'{prefix}uncertainty_score']=float(np.mean(p))
            record[f'{prefix}uncertainty_share']=float(np.mean(p>0.5))
            if section=='full': record['n_sentences']=len(p)
    results.append(record)

scores_df = pd.DataFrame(results)
meta = df[['transcript_id','company_name','event_date','sector','industry']].drop_duplicates(subset=['transcript_id'])
scores_df = scores_df.merge(meta, on='transcript_id', how='left')

print('Computing lexicon baseline...')
scores_df['lexicon_uncertainty_score'] = df.set_index('transcript_id').loc[
    scores_df['transcript_id'].values, 'management_text'].apply(lexicon_score_text).values

output_path = os.path.join(DATA_DIR, 'transcript_scores.parquet')
scores_df.to_parquet(output_path, index=False, engine='pyarrow')
print(f'\nSaved {len(scores_df)} transcript scores to {output_path}')

corr = scores_df['uncertainty_score'].corr(scores_df['lexicon_uncertainty_score'])
print(f'Correlation (model vs lexicon): {corr:.4f}')
print(f'Total time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 7: Download results
from google.colab import files
files.download('/content/data/processed/transcript_scores.parquet')
print('Done! Put transcript_scores.parquet in your laptop data/processed/ folder, then run:')
print('  .venv/Scripts/python.exe run_pipeline.py --from 8 --skip-training')